# Document OCR Pipeline — Mobile Deployment Walkthrough

*Part 3 of 4 · From PyTorch fake quant to shippable mobile bundle*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Document-OCR-Pipeline/blob/main/colab/03_ocr_pipeline_mobile.ipynb)

**Open in Google Colab:** https://colab.research.google.com/github/Gaurav14cs17/Document-OCR-Pipeline/blob/main/colab/03_ocr_pipeline_mobile.ipynb

---

Companion to [02 — Quantization](02_ocr_pipeline_quant.ipynb). That notebook quantizes weights in PyTorch. **This one covers what you still need to run OCR on a phone.**

The gap between Colab and phone:

| Path | Compute | RAM |
|------|---------|-----|
| Fake quant (notebook 02) | $\hat{\mathbf{W}} = Q(\mathbf{W})s$ then **fp16 matmul** | Stores int + dequant buffer |
| Real mobile int8 | $\mathbf{y} = \text{int8\_matmul}(\mathbf{x}_q, Q(\mathbf{W}))$ | No fp weight tensor |

| Stage | What you build |
|-------|----------------|
| **1** | Fake quant vs real mobile quant — why Colab quant ≠ phone speed |
| **2** | Mobile size & latency budget — will Florence-2 fit? |
| **3** | Precision plan export — int4 / int8 / fp16 map per layer |
| **4** | Packed int4 weights — 2 weights per byte, ready for native kernels |
| **5** | Operator audit — which ops Florence-2 needs on mobile |
| **6** | ONNX / runtime export — subgraph demo + full-model blockers |
| **7** | Pre & post-processing spec — pad, tokenize, bbox unmap for the app |
| **8** | Mobile bundle — manifest + files you ship in Android/iOS |
| **9** | On-device validation checklist — accuracy, latency, RAM |

Run **[02_ocr_pipeline_quant.ipynb](02_ocr_pipeline_quant.ipynb) first** (or set `LOAD_PREVIOUS_PLAN = False` to build a quick plan here).

**Series:** [01 OCR](01_document_ocr_pipeline.ipynb) → [02 Quant](02_ocr_pipeline_quant.ipynb) → **03 Mobile** → [04 Mobile complete](04_ocr_pipeline_mobile_complete.ipynb)

## 0 — Install & config

Set platform budgets before export. Typical mid-range phone targets:

$$
\text{disk} \le 150\,\text{MB}, \quad \text{RAM}_{\text{peak}} \le 512\,\text{MB}, \quad T_{\text{page}} \le 8\,\text{s}
$$

**Steps:**

1. Set **`TARGET_PLATFORM`** (`android` | `ios` | `both`).
2. Set **`RUNTIME`** — recommended runtime for that platform.
3. Run install, then **Run all**.

In [ ]:
import os
import re
import subprocess
import sys

# ── CONFIG ───────────────────────────────────────────────
TARGET_PLATFORM = "android"     # android | ios | both
RUNTIME = "onnxruntime"         # onnxruntime | executorch | tflite | coreml
MODEL_ID = "microsoft/Florence-2-base-ft"
TASK = "detect"
EXPORT_DIR = "mobile_bundle"   # output folder to zip and copy to your app
LOAD_PREVIOUS_PLAN = True       # reuse logic from quant notebook if you ran it

# Mobile budget targets (typical mid-range phone)
TARGET_MODEL_MB = 150          # max model size on disk
TARGET_LATENCY_SEC = 8.0       # max seconds per page OCR
TARGET_RAM_MB = 512              # peak RAM budget
MAX_NEW_TOKENS_MOBILE = 256    # shorter generation on phone

FP16_SENSITIVE_PCT = 15
INT8_MID_PCT = 35
MIN_PARAMS_TO_QUANT = 4096
ALWAYS_FP16_PATTERNS = (
    "lm_head", "embed", "pooler", "classifier",
    "visual_projection", "image_projection", "vision_tower",
)
MAX_CALIB_BATCHES = 4

def _pip_version(package):
    r = subprocess.run(
        [sys.executable, "-m", "pip", "show", package],
        capture_output=True, text=True, check=False,
    )
    m = re.search(r"^Version: (.+)$", r.stdout, re.M)
    return m.group(1) if m else ""

def ensure_transformers():
    ok = lambda v: v.startswith("4.49")
    if not ok(_pip_version("transformers")):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "--force-reinstall", "transformers==4.49.0",
        ])
    import transformers
    if not ok(transformers.__version__):
        print("Restart runtime and Run all.")
        os.kill(os.getpid(), 9)
    return transformers.__version__

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "torch", "onnx", "onnxruntime", "pillow", "matplotlib", "requests", "huggingface_hub",
])
print(f"Platform={TARGET_PLATFORM}  runtime={RUNTIME}  transformers {ensure_transformers()}")

---
## Stage 1 — Fake quant vs real mobile quant

### Fake quant (notebook 02)

$$
\mathbf{y} = \mathbf{x} \cdot \underbrace{(Q(\mathbf{W}) \cdot s)^\top}_{\text{dequant to fp16}} + \mathbf{b}
$$

Memory: stores $Q(\mathbf{W})$ (compact) but **compute** uses fp16 GEMM — no speed gain on phone.

### Real int8 (target)

$$
\mathbf{y} \approx (s_x s_w) \cdot \text{GEMM}_{\text{int8}}\!\bigl(Q_x(\mathbf{x}),\, Q(\mathbf{W})\bigr) + \mathbf{b}
$$

**Theorem (first-order equivalence):** if $|x - s_x Q_x(x)| \le \epsilon_x$ and $|w - s_w Q(w)| \le \epsilon_w$, then output error:

$$
|\Delta y| \le \|\mathbf{x}\|_1 \epsilon_w + \|\mathbf{w}\|_1 \epsilon_x + O(\epsilon_x \epsilon_w)
$$

**Proof:** triangle inequality on $\mathbf{x}\hat{\mathbf{w}}^\top - \mathbf{x}\mathbf{w}^\top = \mathbf{x}(\hat{\mathbf{w}}-\mathbf{w})^\top + (\mathbf{x}-\hat{\mathbf{x}})\hat{\mathbf{w}}^\top$.

Below: compare fp16 baseline, fake quant, simulated int8 on one layer.

In [ ]:
import time
import json
import math
import zipfile
import shutil
from dataclasses import dataclass, asdict
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


def qmax_for_bits(n_bits):
    return 2 ** (n_bits - 1) - 1


def symmetric_quantize_per_channel(W, n_bits=8):
    qmax = qmax_for_bits(n_bits)
    W = W.float()
    max_abs = W.abs().amax(dim=1).clamp(min=1e-8)
    scales = max_abs / qmax
    q = torch.round(W / scales.unsqueeze(1)).clamp(-qmax - 1, qmax)
    return q.to(torch.int8), scales


def fake_quant_linear(x, weight, bias, n_bits=8):
    """Colab quant notebook path: dequant weights, float matmul."""
    q, s = symmetric_quantize_per_channel(weight, n_bits)
    w_fp = q.float() * s.unsqueeze(1)
    return F.linear(x, w_fp.to(x.dtype), bias)


def mobile_int8_linear(x, weight, bias):
    """Closer to mobile: quantize x too, int matmul, rescale."""
    w_q, w_s = symmetric_quantize_per_channel(weight, 8)
    # per-token activation quant (simplified dynamic quant)
    x_scale = x.abs().amax(dim=-1, keepdim=True).clamp(min=1e-8) / 127.0
    x_q = torch.round(x / x_scale).clamp(-128, 127)
    # int32 accum → float rescale
    y = torch.matmul(x_q.float(), w_q.float().t())
    y = y * x_scale * w_s.unsqueeze(0)
    if bias is not None:
        y = y + bias
    return y


# Demo on a random layer-sized tensor
in_f, out_f = 768, 768
W = torch.randn(out_f, in_f, device=DEVICE)
b = torch.zeros(out_f, device=DEVICE)
X = torch.randn(32, in_f, device=DEVICE)  # batch of tokens

with torch.no_grad():
    y_fp = F.linear(X, W, b)
    t0 = time.perf_counter()
    for _ in range(100):
        _ = fake_quant_linear(X, W, b, 8)
    fake_ms = (time.perf_counter() - t0) / 100 * 1000
    t0 = time.perf_counter()
    for _ in range(100):
        _ = mobile_int8_linear(X, W, b)
    mobile_ms = (time.perf_counter() - t0) / 100 * 1000
    y_fake = fake_quant_linear(X, W, b, 8)
    y_mob = mobile_int8_linear(X, W, b)

print("Same layer, three paths:")
print(f"  Output MSE fake-quant vs fp16 : {(y_fp - y_fake).pow(2).mean().item():.2e}")
print(f"  Output MSE mobile-int8 vs fp16: {(y_fp - y_mob).pow(2).mean().item():.2e}")
print(f"  Latency fake-quant (still float matmul): {fake_ms:.2f} ms")
print(f"  Latency mobile-int8 sim:                 {mobile_ms:.2f} ms")
print("\n→ Fake quant saves disk space but NOT compute. Mobile needs int kernels + export.")

---
## Stage 2 — Size & latency budget

**Disk size** for $P$ parameters at bit width $b$:

$$
\text{MB} = \frac{P \cdot b}{8 \cdot 10^6}
$$

**KV cache** ($L$ layers, $K$ tokens, hidden $d$, fp16 = 2 bytes):

$$
\text{RAM}_{\text{KV}} = 2 \cdot L \cdot K \cdot d \cdot 2 \quad \text{(keys + values)}
$$

**Proof (KV growth):** autoregressive decode stores $\mathbf{K}_\ell, \mathbf{V}_\ell \in \mathbb{R}^{K \times d}$ per layer; each new token appends one row → memory $\propto K$ linearly.

**Latency bound:** $T_{\text{page}} \approx T_{\text{vision}} + K \cdot T_{\text{step}}$. Compare to `TARGET_MODEL_MB`, `TARGET_RAM_MB`, `TARGET_LATENCY_SEC`.

In [ ]:
ensure_transformers()
from transformers import AutoProcessor, AutoModelForCausalLM

PROMPT = "<OCR_WITH_REGION>" if TASK == "detect" else "<OCR>"
dtype = torch.float16 if DEVICE == "cuda" else torch.float32

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, torch_dtype=dtype, attn_implementation="eager",
).to(DEVICE)
model.eval()


def count_params(module):
    return sum(p.numel() for p in module.parameters())


def estimate_size_mb(n_params, bits):
    return n_params * bits / 8 / 1024 / 1024


total = count_params(model)
linear_only = sum(m.weight.numel() for m in model.modules() if isinstance(m, nn.Linear))
non_linear = total - linear_only

scenarios = {
    "fp16 (no quant)": estimate_size_mb(total, 16),
    "Linear int8 + rest fp16": estimate_size_mb(linear_only, 8) + estimate_size_mb(non_linear, 16),
    "Mixed int4/int8/fp16 (typical plan)": (
        estimate_size_mb(int(linear_only * 0.50), 4)
        + estimate_size_mb(int(linear_only * 0.35), 8)
        + estimate_size_mb(int(linear_only * 0.15), 16)
        + estimate_size_mb(non_linear, 16)
    ),
    "All linear int4 + rest fp16": estimate_size_mb(linear_only, 4) + estimate_size_mb(non_linear, 16),
}

print(f"Model: {MODEL_ID}")
print(f"Total params: {total/1e6:.1f}M  (Linear: {linear_only/1e6:.1f}M)\n")
print(f"{'Scenario':<40} {'Size (MB)':>10}  Fits {TARGET_MODEL_MB}MB?")
print("-" * 65)
for name, mb in scenarios.items():
    ok = "✓" if mb <= TARGET_MODEL_MB else "✗"
    print(f"{name:<40} {mb:>9.1f}  {ok}")

# KV cache estimate during generation
hidden = getattr(model.config, "hidden_size", 768)
n_layers = getattr(model.config, "num_hidden_layers", 12)
kv_mb = MAX_NEW_TOKENS_MOBILE * n_layers * 2 * hidden * 2 / 1024 / 1024  # fp16 K+V
print(f"\nKV cache @ {MAX_NEW_TOKENS_MOBILE} tokens (fp16, rough): ~{kv_mb:.0f} MB RAM during generate")
print(f"Peak RAM budget: {TARGET_RAM_MB} MB  →  {'OK' if kv_mb < TARGET_RAM_MB * 0.4 else 'TIGHT — reduce max tokens or quant KV'}")

---
## Stage 3 — Export precision plan (int4 / int8 / fp16 per layer)

The mobile app needs a JSON map $\pi : \ell \mapsto b_\ell$ where $b_\ell \in \{4,8,16\}$.

We scan sensitivity (same logic as notebook 02) and write `precision_plan.json`:

$$
\pi(\ell) = \begin{cases} 16 & \ell \in \mathcal{P} \cup \text{top-}p\% \\ 8 & \ell \in \text{mid-}q\% \\ 4 & \text{otherwise} \end{cases}
$$

where $\mathcal{P}$ = protected patterns (`lm_head`, embed, vision projection).

**Proof (protected layers):** vision projection error $\Delta \mathbf{h}$ at embedding boundary propagates to all $K$ decode steps; keeping $\mathcal{P}$ at fp16 bounds $\|\Delta \mathbf{h}\|_2 \le \epsilon$ independent of quant depth.

In [ ]:
def pad_info(image):
    w, h = image.size
    side = max(w, h)
    canvas = Image.new("RGB", (side, side), "white")
    pad_x, pad_y = (side - w) // 2, (side - h) // 2
    canvas.paste(image, (pad_x, pad_y))
    return canvas, pad_x, pad_y, w, h


def load_sample_image():
    try:
        url = "https://raw.githubusercontent.com/Gaurav14cs17/Document-OCR-Pipeline/main/assets/table_page.png"
        return Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB")
    except Exception:
        img = Image.new("RGB", (640, 480), "white")
        ImageDraw.Draw(img).text((20, 20), "Sample", fill="black")
        return img


def layer_matches(name, patterns):
    n = name.lower()
    return any(p in n for p in patterns)


def measure_output_mse(layer, inputs, n_bits):
    if inputs is None or inputs.numel() == 0:
        return 0.0
    q, s = symmetric_quantize_per_channel(layer.weight.data, n_bits)
    w_hat = q.float() * s.unsqueeze(1)
    x = inputs[:1024].to(layer.weight.device)
    with torch.no_grad():
        y0 = F.linear(x, layer.weight.float(), layer.bias)
        y1 = F.linear(x, w_hat.to(x.dtype), layer.bias)
        return (y0 - y1).pow(2).mean().item()


@dataclass
class LayerPlanEntry:
    name: str
    shape: list
    num_params: int
    bits: str
    sensitivity: float
    output_mse_int4: float
    output_mse_int8: float
    note: str


def build_precision_plan(model, processor, image, prompt, n_calib=4):
    linears = [(n, m) for n, m in model.named_modules() if isinstance(m, nn.Linear)]
    captures = {n: [] for n, _ in linears}
    handles = []

    def hook(name):
        def fn(_m, inp, _o):
            x = inp[0].detach()
            if x.dim() == 3:
                x = x.reshape(-1, x.shape[-1])
            captures[name].append(x.cpu())
        return fn

    mod_map = dict(model.named_modules())
    for n, _ in linears:
        handles.append(mod_map[n].register_forward_hook(hook(n)))

    padded, *_ = pad_info(image)
    for _ in range(n_calib):
        inputs = processor(text=prompt, images=padded, return_tensors="pt").to(DEVICE)
        inputs["pixel_values"] = inputs["pixel_values"].to(dtype=next(model.parameters()).dtype)
        with torch.no_grad():
            model.generate(**{k: inputs[k] for k in ("input_ids", "pixel_values")},
                             max_new_tokens=32, num_beams=1, use_cache=False)
    for h in handles:
        h.remove()

    entries = []
    for name, mod in linears:
        inp = torch.cat(captures[name], dim=0) if captures[name] else torch.empty(0)
        out4 = measure_output_mse(mod, inp, 4)
        out8 = measure_output_mse(mod, inp, 8)
        act_max = float(inp.abs().amax()) if inp.numel() else 0.0
        sens = out4 * (1 + 0.1 * math.log1p(act_max))
        protected = layer_matches(name, ALWAYS_FP16_PATTERNS) or mod.weight.numel() < MIN_PARAMS_TO_QUANT
        entries.append(LayerPlanEntry(
            name=name, shape=list(mod.weight.shape), num_params=mod.weight.numel(),
            bits="fp16", sensitivity=sens, output_mse_int4=out4, output_mse_int8=out8,
            note="protected" if protected else "pending",
        ))

    entries.sort(key=lambda e: e.sensitivity, reverse=True)
    candidates = [e for e in entries if e.note != "protected"]
    n = len(candidates)
    n_fp16 = max(1, round(n * FP16_SENSITIVE_PCT / 100))
    n_int8 = round(n * INT8_MID_PCT / 100)
    for i, e in enumerate(candidates):
        if i < n_fp16:
            e.bits, e.note = "fp16", "high sensitivity — keep on phone"
        elif i < n_fp16 + n_int8:
            e.bits, e.note = "int8", "medium — good mobile balance"
        else:
            e.bits, e.note = "int4", "low sensitivity — max compression"
    for e in entries:
        if e.note == "protected":
            e.bits, e.note = "fp16", "lm_head/embed/vision — never quantize on mobile"
    return entries


image = load_sample_image()
plan = build_precision_plan(model, processor, image, PROMPT, MAX_CALIB_BATCHES)

from collections import Counter
c = Counter(e.bits for e in plan)
print(f"Precision plan: {dict(c)}")
print(f"\n{'Layer':<42} {'Bits':<6} {'Sens':>8}  Note")
print("-" * 85)
for e in plan[:12]:
    short = e.name.split(".")[-1]
    print(f"{short:<42} {e.bits:<6} {e.sensitivity:>8.2e}  {e.note}")

---
## Stage 4 — Pack int4 weights (2 weights per byte)

Values $q \in \mathcal{Q}_4 = \{-8,-7,\ldots,7\}$ (16 states = 4 bits). Pack pairs into one byte:

$$
B_k = (q_{2k+1} + 8) \cdot 16 + (q_{2k} + 8) \in [0,255]
$$

### Bijection proof

Define $\phi: \mathcal{Q}_4 \times \mathcal{Q}_4 \to \{0,\ldots,255\}$ by $\phi(q_{\text{hi}}, q_{\text{lo}}) = (q_{\text{hi}}+8)\cdot 16 + (q_{\text{lo}}+8)$.

**Claim:** $\phi$ is bijective.

**Proof:** map $q+8 \in \{0,\ldots,15\}$ to 4-bit nibble. High nibble $\times 16$ + low nibble uniquely encodes two nibbles in one byte. Inverse: $q_{\text{lo}} = (B_k \bmod 16) - 8$, $q_{\text{hi}} = \lfloor B_k/16 \rfloor - 8$.

**Storage:** $OI/2$ bytes vs $OI$ bytes for int8 → **2× compression** vs int8, **4×** vs fp16 (ignoring scales).

In [ ]:
def pack_int4_tensor(q_int8: torch.Tensor) -> bytes:
    """Pack signed int4 values (stored in int8) into bytes, 2 nibbles each."""
    flat = q_int8.flatten().cpu().numpy().astype(np.int8)
    # map to 0..15 for unsigned nibble storage
    flat = flat & 0x0F
    if len(flat) % 2 == 1:
        flat = np.append(flat, 0)
    hi = flat[0::2]
    lo = flat[1::2]
    packed = ((hi << 4) | lo).astype(np.uint8)
    return packed.tobytes()


def unpack_int4_tensor(data: bytes, numel: int) -> torch.Tensor:
    arr = np.frombuffer(data, dtype=np.uint8)
    hi = (arr >> 4) & 0x0F
    lo = arr & 0x0F
    vals = np.empty(numel, dtype=np.int8)
    vals[0::2] = hi[: (numel + 1) // 2]
    vals[1::2] = lo[: numel // 2]
    # sign-extend 4-bit
    vals = np.where(vals >= 8, vals - 16, vals)
    return torch.from_numpy(vals.astype(np.int8))


def export_layer_weights(layer: nn.Linear, bits: str, out_dir: Path, layer_name: str):
    out_dir.mkdir(parents=True, exist_ok=True)
    safe = layer_name.replace(".", "__")
    meta = {"name": layer_name, "shape": list(layer.weight.shape), "bits": bits}

    if bits == "fp16":
        path = out_dir / f"{safe}.fp16.bin"
        layer.weight.detach().cpu().half().numpy().tofile(path)
        meta["file"] = path.name
        meta["bytes"] = path.stat().st_size
    elif bits in ("int8", "int4"):
        n_bits = int(bits.replace("int", ""))
        q, scales = symmetric_quantize_per_channel(layer.weight.data, n_bits)
        if bits == "int4":
            wpath = out_dir / f"{safe}.int4.bin"
            wpath.write_bytes(pack_int4_tensor(q))
        else:
            wpath = out_dir / f"{safe}.int8.bin"
            q.cpu().numpy().tofile(wpath)
        spath = out_dir / f"{safe}.scales.bin"
        scales.cpu().numpy().astype(np.float32).tofile(spath)
        meta.update({"file": wpath.name, "scales_file": spath.name, "bytes": wpath.stat().st_size + spath.stat().st_size})
        if layer.bias is not None:
            bpath = out_dir / f"{safe}.bias.bin"
            layer.bias.detach().cpu().float().numpy().tofile(bpath)
            meta["bias_file"] = bpath.name
    return meta


# Demo pack/unpack roundtrip on first int4 layer in plan
int4_entry = next((e for e in plan if e.bits == "int4"), plan[0])
mod = dict(model.named_modules())[int4_entry.name]
q, s = symmetric_quantize_per_channel(mod.weight.data, 4)
packed = pack_int4_tensor(q)
q_back = unpack_int4_tensor(packed, q.numel()).reshape(q.shape)
print(f"Layer: {int4_entry.name}  bits=int4")
print(f"  Unpacked int8 storage: {q.numel()} bytes")
print(f"  Packed int4 storage:   {len(packed)} bytes  ({100*len(packed)/q.numel():.0f}% of int8)")
print(f"  Pack roundtrip match:  {torch.equal(q.cpu(), q_back)}")

---
## Stage 5 — Operator audit (what the mobile runtime must support)

Forward graph $G = (V, E)$ where vertices $v \in V$ are ops. Mobile runtime supports $V_{\text{m}} \subset V$.

**Unsupported op** $v \notin V_{\text{m}}$ requires graph surgery:

$$
G \xrightarrow{\text{partition}} G_{\text{device}} \sqcup G_{\text{fallback}}
$$

where $G_{\text{device}}$ runs on NPU/GPU and $G_{\text{fallback}}$ on CPU fp16.

We enumerate op types in Florence-2 and flag $V \setminus V_{\text{m}}$ (dynamic shapes, custom attention, vision-specific ops).

In [ ]:
MOBILE_RISKY_OPS = {
    "aten::einsum", "aten::grid_sampler", "aten::inverse",
    "aten::linalg", "aten::complex", "aten::fft",
}
MOBILE_OK_OPS = {
    "aten::linear", "aten::matmul", "aten::add", "aten::mul",
    "aten::layer_norm", "aten::softmax", "aten::gelu", "aten::relu",
    "aten::conv2d", "aten::reshape", "aten::transpose",
}

op_counts = {}
for name, mod in model.named_modules():
    op = type(mod).__name__
    op_counts[op] = op_counts.get(op, 0) + 1

print("Module types in Florence-2 (top 20):")
for op, cnt in sorted(op_counts.items(), key=lambda x: -x[1])[:20]:
    risk = ""
    if op in ("Florence2Attention", "Florence2VisionModel", "Florence2LanguageModel"):
        risk = " ← custom HuggingFace code"
    print(f"  {op:<35} {cnt:>4}{risk}")

blockers = [
    "trust_remote_code — custom Python forward, not portable as-is",
    "Autoregressive generate() loop — must reimplement in app or export as loop",
    "Vision encoder + language decoder — two graphs or one big graph",
    "Dynamic sequence length — complicates TFLite; ONNX supports better",
    "post_process_generation — OCR bbox parsing stays in app code",
]
print("\nFull-model mobile blockers:")
for b in blockers:
    print(f"  • {b}")

runtime_notes = {
    "onnxruntime": "Best starting point. ONNX Runtime Mobile on Android/iOS. Needs op set audit.",
    "executorch": "PyTorch-native mobile path. Good if you stay in PyTorch ecosystem.",
    "tflite": "Strong on Android NPU. Florence-2 conversion is hard (custom ops).",
    "coreml": "iOS only. Needs coremltools; transformer support improving.",
}
print(f"\nRecommended for {TARGET_PLATFORM}: {runtime_notes.get(RUNTIME, RUNTIME)}")

---
## Stage 6 — ONNX export demo (single Linear layer)

Full Florence-2 ONNX export is non-trivial (dynamic seq len, vision encoder). We export **one quant-ready Linear** as template:

$$
\text{ONNX}(\mathbf{x}) = \mathbf{x} \cdot \text{Dequant}(Q(\mathbf{W}))^\top + \mathbf{b}
$$

Your converter pipeline repeats this pattern per layer, then fuses int8 kernels where the runtime allows.

In [ ]:
class ExportableInt8Linear(nn.Module):
    """ONNX-friendly: explicit quant → matmul → dequant."""

    def __init__(self, linear: nn.Linear):
        super().__init__()
        q, s = symmetric_quantize_per_channel(linear.weight.data, 8)
        self.register_buffer("weight_q", q)
        self.register_buffer("weight_scale", s)
        self.bias = nn.Parameter(linear.bias.clone()) if linear.bias is not None else None

    def forward(self, x):
        w = self.weight_q.float() * self.weight_scale.unsqueeze(1)
        return F.linear(x, w.to(x.dtype), self.bias)


sample_linear = next(m for n, m in model.named_modules() if isinstance(m, nn.Linear))
export_mod = ExportableInt8Linear(sample_linear).cpu().eval()
dummy_x = torch.randn(1, sample_linear.in_features)
onnx_path = Path(EXPORT_DIR) / "demo_linear_int8.onnx"
onnx_path.parent.mkdir(parents=True, exist_ok=True)

try:
    torch.onnx.export(
        export_mod, dummy_x, str(onnx_path),
        input_names=["input"], output_names=["output"],
        dynamic_axes={"input": {0: "batch", 1: "features"}, "output": {0: "batch"}},
        opset_version=17,
    )
    import onnxruntime as ort
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    ort_out = sess.run(None, {"input": dummy_x.numpy()})[0]
    pt_out = export_mod(dummy_x).detach().numpy()
    print(f"Exported: {onnx_path}  ({onnx_path.stat().st_size/1024:.1f} KB)")
    print(f"ONNX vs PyTorch max diff: {np.abs(ort_out - pt_out).max():.2e}")
except Exception as e:
    print(f"ONNX export demo failed: {e}")
    print("Install/onnx version mismatch is common — pattern still applies for your CI pipeline.")

---
## Stage 7 — Pre & post-processing spec for the mobile app

Native app must implement the same transform chain as notebook 01:

$$
\text{JSON} = \mathcal{U} \circ \mathcal{G}_\tau \circ \mathcal{D} \circ \mathcal{G}_\theta \circ \mathcal{P}(\mathbf{I})
$$

where $\mathcal{P}$ = pad + normalize, $\mathcal{U}$ = unmap + clean.

### Pad (must match exactly)

$$
S = \max(W_0, H_0), \quad \text{pad}_x = \lfloor(S-W_0)/2\rfloor
$$

### Unmap (inverse)

$$
(x', y') = (x - \text{pad}_x,\; y - \text{pad}_y)
$$

**Proof (cross-platform correctness):** if native $\mathcal{P}' \neq \mathcal{P}$, then $\mathcal{U}' \circ \mathcal{G}_\theta \circ \mathcal{P}' \neq \mathcal{U} \circ \mathcal{G}_\theta \circ \mathcal{P}$ even with identical weights — bbox drift without model error.

We export `preprocess.json` / `postprocess.json` with these formulas.

In [ ]:
preprocess_spec = {
    "steps": [
        {"id": "load_image", "desc": "Camera or gallery → RGB bitmap"},
        {"id": "pad_square", "desc": "Pad to max(w,h) square, white background, track pad_x/pad_y"},
        {"id": "normalize", "desc": "Use Florence-2 processor mean/std — copy from processor config"},
        {"id": "tokenize", "desc": "Prompt: '<OCR_WITH_REGION>' for detect, '<OCR>' for text"},
        {"id": "build_tensors", "desc": "input_ids [1, seq], pixel_values [1, 3, H, W]"},
    ],
    "pad_formula": {
        "side": "max(width, height)",
        "pad_x": "(side - width) // 2",
        "pad_y": "(side - height) // 2",
    },
    "prompts": {"detect": "<OCR_WITH_REGION>", "ocr": "<OCR>"},
    "max_new_tokens_mobile": MAX_NEW_TOKENS_MOBILE,
}

postprocess_spec = {
    "steps": [
        {"id": "decode_tokens", "desc": "Tokenizer decode generated token IDs → raw string"},
        {"id": "parse_regions", "desc": "Parse Florence-2 region format (quad_boxes + labels)"},
        {"id": "quad_to_bbox", "desc": "8-point quad → [x1,y1,x2,y2] on padded image"},
        {"id": "unmap_bbox", "desc": "Subtract pad_x/pad_y to map back to original image coords"},
        {"id": "clean_html", "desc": "Strip <p> tags from labels if present"},
    ],
    "unmap_formula": {
        "x1": "clamp(x1 - pad_x, 0, orig_w)",
        "y1": "clamp(y1 - pad_y, 0, orig_h)",
    },
    "output_format": {
        "detect": {"text_lines": [{"text": "str", "bbox": ["x1","y1","x2","y2"]}]},
        "ocr": {"text": "str", "html": "str"},
    },
}

print("PREPROCESS (app must implement):")
for s in preprocess_spec["steps"]:
    print(f"  {s['id']}: {s['desc']}")
print("\nPOSTPROCESS (app must implement):")
for s in postprocess_spec["steps"]:
    print(f"  {s['id']}: {s['desc']}")

# Demo unmap on sample
padded, px, py, ow, oh = pad_info(image)
print(f"\nExample: 640×480 image → padded {padded.size[0]}×{padded.size[1]}, pad_x={px}, pad_y={py}")

---
## Stage 8 — Build the mobile bundle (files to ship in your app)

We write everything into `mobile_bundle/`:
- `manifest.json` — model info, runtime, platform
- `precision_plan.json` — per-layer bit width $\pi(\ell)$
- `preprocess.json` / `postprocess.json` — app integration spec
- `weights/` — packed int4/int8/fp16 binaries (sample layers)
- `tokenizer/` — vocab config pointers

**Ship condition:** $\sum \text{bytes}_\ell \le \text{TARGET\_MODEL\_MB} \cdot 10^6$.

In [ ]:
bundle = Path(EXPORT_DIR)
if bundle.exists():
    shutil.rmtree(bundle)
weights_dir = bundle / "weights"
weights_dir.mkdir(parents=True)

# Export weights for quant layers (cap at 10 for Colab speed)
weight_manifest = []
mod_map = dict(model.named_modules())
for entry in [e for e in plan if e.bits != "fp16"][:10]:
    meta = export_layer_weights(mod_map[entry.name], entry.bits, weights_dir, entry.name)
    weight_manifest.append(meta)

precision_plan = {
    "model_id": MODEL_ID,
    "generated_by": "ocr_pipeline_mobile.ipynb",
    "layers": [asdict(e) for e in plan],
    "summary": dict(Counter(e.bits for e in plan)),
}
(bundle / "precision_plan.json").write_text(json.dumps(precision_plan, indent=2))
(bundle / "preprocess.json").write_text(json.dumps(preprocess_spec, indent=2))
(bundle / "postprocess.json").write_text(json.dumps(postprocess_spec, indent=2))

manifest = {
    "model_id": MODEL_ID,
    "task": TASK,
    "target_platform": TARGET_PLATFORM,
    "runtime": RUNTIME,
    "budget": {
        "max_model_mb": TARGET_MODEL_MB,
        "max_latency_sec": TARGET_LATENCY_SEC,
        "max_ram_mb": TARGET_RAM_MB,
    },
    "files": {
        "precision_plan": "precision_plan.json",
        "preprocess": "preprocess.json",
        "postprocess": "postprocess.json",
        "demo_onnx": "demo_linear_int8.onnx",
        "weights_dir": "weights/",
    },
    "weight_exports_sample": weight_manifest,
    "next_steps": [
        "Export full vision + language graph to ONNX/CoreML/TFLite",
        "Replace fake-quant forward with int8 kernels per layer plan",
        "Bundle tokenizer vocab in app assets",
        "Implement preprocess/postprocess in Kotlin or Swift",
        "Run on-device checklist (Stage 9)",
    ],
}
(bundle / "manifest.json").write_text(json.dumps(manifest, indent=2))

zip_path = Path(f"{EXPORT_DIR}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in bundle.rglob("*"):
        if f.is_file():
            zf.write(f, f.relative_to(bundle.parent))

total_bytes = sum(f.stat().st_size for f in bundle.rglob("*") if f.is_file())
print(f"Bundle written to ./{EXPORT_DIR}/")
print(f"Zip: ./{zip_path}  ({zip_path.stat().st_size/1024:.1f} KB)")
print(f"\nContents:")
for f in sorted(bundle.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(bundle)}  ({f.stat().st_size/1024:.1f} KB)")

---
## Stage 9 — On-device validation checklist

Run these on a **real phone** after integrating the bundle. Colab can't measure thermal throttling or NPU driver bugs.

**Acceptance metrics:**
- $\text{IoU}_{\text{bbox}} \ge 0.85$ vs fp16 baseline on test set
- $T_{\text{page}} \le \text{TARGET\_LATENCY\_SEC}$
- $\text{RAM}_{\text{peak}} \le \text{TARGET\_RAM\_MB}$

Copy the checklist into your QA doc.

In [ ]:
checklist = [
    {"category": "Accuracy", "item": "Same test image on PC vs phone — text matches?", "pass": "TODO"},
    {"category": "Accuracy", "item": "Bounding boxes align on detect task (within 5px)?", "pass": "TODO"},
    {"category": "Accuracy", "item": "Hindi + English mixed pages still readable?", "pass": "TODO"},
    {"category": "Latency", "item": f"Full page OCR < {TARGET_LATENCY_SEC}s on mid-range device?", "pass": "TODO"},
    {"category": "Latency", "item": "Cold start (first inference) acceptable?", "pass": "TODO"},
    {"category": "Memory", "item": f"Peak RAM < {TARGET_RAM_MB} MB during generate?", "pass": "TODO"},
    {"category": "Memory", "item": "No OOM on 4GB RAM phone?", "pass": "TODO"},
    {"category": "Size", "item": f"APK/IPA + model < {TARGET_MODEL_MB} MB budget?", "pass": "TODO"},
    {"category": "Battery", "item": "10 pages OCR — battery drain acceptable?", "pass": "TODO"},
    {"category": "Stability", "item": "100 consecutive runs — no crash/leak?", "pass": "TODO"},
    {"category": "Numerics", "item": "int8 phone output vs fp16 reference — MSE within threshold?", "pass": "TODO"},
]

platform_integration = {
    "android": {
        "runtime": "ONNX Runtime Mobile or ExecuTorch",
        "delegate": "NNAPI / GPU delegate for speed",
        "language": "Kotlin + JNI for native inference",
        "assets": "Copy mobile_bundle/ to app/src/main/assets/",
    },
    "ios": {
        "runtime": "ONNX Runtime Mobile or Core ML",
        "delegate": "Apple Neural Engine via Core ML",
        "language": "Swift + ONNXRuntime Obj-C API",
        "assets": "Add mobile_bundle/ to Xcode Copy Bundle Resources",
    },
}

print("ON-DEVICE VALIDATION CHECKLIST")
print("=" * 70)
for row in checklist:
    print(f"[{row['pass']}] {row['category']:<10} {row['item']}")

print(f"\n{TARGET_PLATFORM.upper()} integration notes:")
for k, v in platform_integration.get(TARGET_PLATFORM, platform_integration["android"]).items():
    print(f"  {k}: {v}")

# Save checklist into bundle
(Path(EXPORT_DIR) / "validation_checklist.json").write_text(
    json.dumps({"checklist": checklist, "platform": platform_integration}, indent=2)
)
print(f"\nSaved validation_checklist.json into ./{EXPORT_DIR}/")

---
## Summary — the full mobile path

```
02_ocr_pipeline_quant.ipynb  →  quantize weights, pick int4/int8/fp16 per layer
         ↓
03_ocr_pipeline_mobile.ipynb →  export plan, pack weights, ONNX demo, app spec
         ↓
Your Android/iOS app         →  load bundle, int8 inference, preprocess/postprocess
         ↓
Real phone QA                →  Stage 9 checklist
```

**End-to-end compression ratio** (mixed plan):

$$
\rho = \frac{\sum_\ell O_\ell I_\ell \cdot 16}{\sum_\ell O_\ell I_\ell \cdot b_\ell} \approx 2\text{–}4\times
$$

| Gap | Covered here? | Still manual |
|-----|---------------|-------------|
| Fake quant → real int8 | Stage 1 demo | Full graph int8 kernels |
| Which layers int4/int8/fp16 | Stage 3 `precision_plan.json` | Tune percentages |
| Packed int4 on disk | Stage 4 `.bin` files | Native loader in app |
| Runtime export | Stage 6 ONNX demo layer | Full Florence-2 export |
| App preprocess/postprocess | Stage 7 JSON spec | Kotlin/Swift code |
| Ship bundle | Stage 8 zip | CI + app integration |
| Phone testing | Stage 9 checklist | Real devices |

**Next:** [04 — Complete Mobile VLM Pipeline](04_ocr_pipeline_mobile_complete.ipynb)